In [5]:
from gbp import gbp
from gbp.factors import linear_displacement
import numpy as np

np.random.seed(0)

In [6]:
class args:
    dim = 2
    M = 3
    n_varnodes = 20
    gauss_noise_std = 10.
    n_iters = 50
    loop_graph = False


In [7]:
# Create priors
priors_mu = np.random.rand(args.n_varnodes, args.dim)  # grid goes from 0 to 10 along x and y axis
priors_mu[0] = [20, 80]

for i in range(1, args.n_varnodes):
    priors_mu[i] = priors_mu[i - 1] - [np.random.uniform(0, 1), np.random.uniform(0, 1)]

priors_mu = priors_mu * 10

prior_sigma = 3 * args.gauss_noise_std * np.eye(args.dim)

prior_lambda = np.linalg.inv(prior_sigma)
priors_lambda = [prior_lambda] * args.n_varnodes
priors_eta = []
for mu in priors_mu:
    priors_eta.append(np.dot(prior_lambda, mu))

print(priors_mu)
print(priors_eta)

# Generate connections between variables
gt_measurements, noisy_measurements = [], []
measurements_nodeIDs = []
num_edges_per_node = np.zeros(args.n_varnodes)
n_edges = 0

for i, mu in enumerate(priors_mu):
    dists = []
    for j, mu1 in enumerate(priors_mu):
        dists.append(np.linalg.norm(mu - mu1))
    for j in np.array(dists).argsort()[1:args.M + 1]:  # As closest node is itself
        mu1 = priors_mu[j]
        if [j, i] not in measurements_nodeIDs:  # To avoid double counting
            n_edges += 1
            gt_measurements.append(mu - mu1)
            noisy_measurements.append(mu - mu1 + np.random.normal(0., args.gauss_noise_std, args.dim))
            measurements_nodeIDs.append([i, j])

            num_edges_per_node[i] += 1
            num_edges_per_node[j] += 1


def generate_noisy_measurement(mu1, mu2):
    return mu1 - mu2 + np.random.normal(0., args.gauss_noise_std, args.dim)


graph = gbp.FactorGraph(nonlinear_factors=False)

# Initialize variable nodes for frames with prior
for i in range(args.n_varnodes):
    new_var_node = gbp.VariableNode(i, args.dim)
    new_var_node.prior.eta = priors_eta[i]
    new_var_node.prior.lam = priors_lambda[i]
    graph.var_nodes.append(new_var_node)

for f, measurement in enumerate(noisy_measurements):
    new_factor = gbp.Factor(f,
                            [graph.var_nodes[measurements_nodeIDs[f][0]], graph.var_nodes[measurements_nodeIDs[f][1]]],
                            measurement,
                            args.gauss_noise_std,
                            linear_displacement.meas_fn,
                            linear_displacement.jac_fn,
                            loss=None,
                            mahalanobis_threshold=2)

    graph.var_nodes[measurements_nodeIDs[f][0]].adj_factors.append(new_factor)
    graph.var_nodes[measurements_nodeIDs[f][1]].adj_factors.append(new_factor)
    graph.factors.append(new_factor)

graph.update_all_beliefs()
graph.compute_all_factors()

graph.n_var_nodes = args.n_varnodes
graph.n_factor_nodes = len(noisy_measurements)
graph.n_edges = 2 * len(noisy_measurements)

print(f'Number of variable nodes {graph.n_var_nodes}')
print(f'Number of edges per variable node {args.M}')
print(f'Number of dofs at each variable node {args.dim}\n')

mu, sigma = graph.joint_distribution_cov()  # Get batch solution

[[200.         800.        ]
 [196.40492099 795.62968046]
 [189.42860903 795.02742575]
 [182.76094188 788.32104705]
 [180.65711627 787.03178407]
 [177.50283276 783.39467636]
 [171.80086506 779.00866123]
 [161.91712668 777.98821312]
 [159.82835911 776.37511794]
 [153.29727586 773.84220192]
 [148.63416813 771.397946  ]
 [147.0444723  770.29419459]
 [140.4811764  768.91236507]
 [138.51535278 765.22511337]
 [130.30542049 764.25410061]
 [121.92597141 763.29311653]
 [112.16137676 758.60660451]
 [102.39376588 752.55814931]
 [ 95.00113008 752.16627139]
 [ 92.17306046 750.96430578]]
[array([ 6.66666667, 26.66666667]), array([ 6.5468307 , 26.52098935]), array([ 6.31428697, 26.50091419]), array([ 6.0920314 , 26.27736823]), array([ 6.02190388, 26.2343928 ]), array([ 5.91676109, 26.11315588]), array([ 5.7266955 , 25.96695537]), array([ 5.39723756, 25.93294044]), array([ 5.32761197, 25.8791706 ]), array([ 5.1099092 , 25.79474006]), array([ 4.95447227, 25.71326487]), array([ 4.90148241, 25.67647315])

In [8]:
import pygame
import random
import sys

# Initialize Pygame
pygame.init()

# Screen dimensions
screen_width = 1000
screen_height = 1000
screen = pygame.display.set_mode((screen_width, screen_height))

# Title and Icon
pygame.display.set_caption("Moving Points with Transparent Circles")

# Color definitions
black = (0, 0, 0)
green = (0, 255, 0, 128)  # Green with half transparency
white = (255, 255, 255)
blue = (0, 0, 255)
red = (255, 0, 0)
orange = (255, 165, 0)


# Function to draw a semi-transparent circle around a point
def draw_transparent_circle(surface, color, center, radius):
    # Create a temporary surface with per-pixel alpha
    temp_surface = pygame.Surface((2 * radius, 2 * radius), pygame.SRCALPHA)
    pygame.draw.circle(temp_surface, color, (radius, radius), radius)
    surface.blit(temp_surface, (center[0] - radius, center[1] - radius))


# Function to draw a semi-transparent ellipse around a point
def draw_transparent_ellipse(surface, color, center, size):
    # Create a temporary surface with per-pixel alpha
    temp_surface = pygame.Surface(size, pygame.SRCALPHA)
    # The ellipse will be drawn inside a rect defined by the size, so the position (0,0) is relative to the surface
    pygame.draw.ellipse(temp_surface, color, (0, 0, size[0], size[1]))
    # Blit this surface onto the main surface, centered on the given coordinates
    surface.blit(temp_surface, (center[0] - size[0] // 2, center[1] - size[1] // 2))


def draw_transparent_line(surface, color, start, end):
    # Create a temporary surface with per-pixel alpha, large enough to contain the line
    # For simplicity, this example uses the size of the entire screen, but you could use a smaller surface if preferred
    temp_surface = pygame.Surface(surface.get_size(), pygame.SRCALPHA)

    # Draw the line on the temporary surface
    pygame.draw.line(temp_surface, color, start, end, 1)

    # Blit this surface onto the main surface
    surface.blit(temp_surface, (0, 0))


def get_line_of_factor(factor):
    mu1 = factor.adj_var_nodes[0].mu
    mu2 = factor.adj_var_nodes[1].mu
    return mu1, mu2


# Function to wait for space bar
def wait_for_space():
    waiting = True
    while waiting:
        for event in pygame.event.get():
            if event.type == pygame.QUIT:
                pygame.quit()
                sys.exit()
            elif event.type == pygame.KEYDOWN:
                if event.key == pygame.K_SPACE:
                    waiting = False


# Main loop
running = True

means, sigmas = graph.joint_distribution_cov()

move = 5

from formation_control.simple_follower import SimpleFollower

x_dist = 20
y_dist = 16
dist_between_followers = np.sqrt(x_dist ** 2 + y_dist ** 2)
followers = []
for i in range(args.n_varnodes):
    followers.append(SimpleFollower(x_dist, y_dist, [move * 20, move * 20]))


def find_points_within_distance_indices_numpy(points, query_point, distance):
    points_array = np.array(points).reshape(-1, 2)
    distances = np.linalg.norm(points_array - np.array(query_point), axis=1)
    indices = np.where(distances <= distance)[0]
    within_distance = points_array[distances <= distance]
    return within_distance, indices


def update_factors(gt_pos):
    priors_eta = []
    graph = gbp.FactorGraph(nonlinear_factors=False)
    for i in range(args.n_varnodes):
        graph.var_nodes.append(gbp.VariableNode(i, args.dim))
        priors_eta.append(np.dot(prior_lambda, gt_pos[i * 2:i * 2 + 2]))
        graph.var_nodes[i].prior.eta = priors_eta[i]
        graph.var_nodes[i].prior.lam = priors_lambda[i]

    for i in range(args.n_varnodes):
        mui = np.array([gt_pos[i * 2], gt_pos[i * 2 + 1]])
        # distances, indices = find_points_within_distance_indices_numpy(gt_pos, mui, dist_between_followers * 10)
        if args.loop_graph:
            indices = []
            n = args.n_varnodes  # Total number of variable nodes
            for j in range(i - args.M, i + args.M + 1):
                wrapped_index = j % n  # Use modular arithmetic to wrap around
                indices.append(wrapped_index)
        else:
            indices = range(max(0, i - args.M), min(args.n_varnodes, i + args.M + 1))
        # connecting to the indices
        for j in indices:
            if i != j:
                muj = np.array([gt_pos[j * 2], gt_pos[j * 2 + 1]])
                new_measurement = generate_noisy_measurement(muj, mui)
                new_factor = gbp.Factor(i * args.n_varnodes + j,
                                        [graph.var_nodes[i], graph.var_nodes[j]],
                                        new_measurement,
                                        args.gauss_noise_std,
                                        linear_displacement.meas_fn,
                                        linear_displacement.jac_fn,
                                        loss=None,
                                        mahalanobis_threshold=2)
                graph.var_nodes[i].adj_factors.append(new_factor)
                graph.var_nodes[j].adj_factors.append(new_factor)
                graph.factors.append(new_factor)

    return graph


while running:
    for event in pygame.event.get():
        if event.type == pygame.QUIT:
            running = False

    means[0] += move
    means[1] -= move

    for i in range(1, args.n_varnodes):
        mu1 = np.array([means[i * 2], means[i * 2 + 1]])
        mu2 = np.array([means[(i - 1) * 2], means[(i - 1) * 2 + 1]])
        # mu1 = graph.var_nodes[i].mu
        # mu2 = graph.var_nodes[0].mu
        measurement = mu2 - mu1
        # measurement = generate_noisy_measurement(mu2, mu1)
        dx, dy = followers[i].follow(measurement)
        means[i * 2] += dx
        means[i * 2 + 1] += dy

    graph = update_factors(means)

    # Fill the screen with black
    screen.fill(black)

    # update pose graph
    for pgo_iter in range(args.n_iters):
        graph.update_all_beliefs()
        graph.compute_all_factors()
        graph.synchronous_iteration()

    mu_estimates = []

    # Move and draw points
    for i in range(args.n_varnodes):
        pos = graph.var_nodes[i].mu
        mu_estimates.append(pos)
        mu = (int(pos[0]), int(pos[1]))
        sigma = np.linalg.inv(np.sqrt(graph.var_nodes[i].belief.lam))
        sigma = np.diag(sigma)
        # Draw the semi-transparent circle around the point
        draw_transparent_ellipse(screen, green, mu, (int(sigma[0] * 5), int(sigma[1] * 5)))

        # Draw the point itself
        pygame.draw.circle(screen, blue, mu, 3)

        # Draw the ground truth
        mu_gt = (int(means[i * 2]), int(means[i * 2 + 1]))

        color = orange if i == 0 else white
        pygame.draw.circle(screen, color, mu_gt, 3)

        draw_transparent_line(screen, red, mu, mu_gt)

    # Draw the factors
    for factor in graph.factors:
        mu1, mu2 = get_line_of_factor(factor)
        mu1 = (int(mu1[0]), int(mu1[1]))
        mu2 = (int(mu2[0]), int(mu2[1]))
        draw_transparent_line(screen, white, mu1, mu2)

    # wait_for_space()

    # Update the display
    pygame.display.flip()

    # Cap the frame rate
    pygame.time.Clock().tick(20)

# Quit Pygame
pygame.quit()


KeyboardInterrupt: 